# Sample-based Krylov Quantum Diagonalization

*Usage estimate: 30 seconds on ibm_fez (NOTE: This is an estimate only. Your runtime might vary.)*

This challenge is based on the [_Quantum-centric Algorithm for Sampled-based Krylov Diagonalization_](https://arxiv.org/pdf/2501.09702) paper. Participants will get hands-on experience with developing an SQKD workflow and executing it on IBM Quantum hardware.

In this challenge, you will implement a workflow for the Sample-Based Krylov Quantum Diagonalization (SKQD) algorithm. First, in [**Part 1**](#part1), you will focus on the __perturbed transverse-field Ising model (TFIM)__. You will implement the algorithm for the TFIM Hamiltonian for a small scale setting of 12 qubits. In [**Part 2**](#part2), you will extend the algorithm to a larger scale setting of 44 qubits for a different Hamiltonian, the __Single-impurity Anderson model (SIAM)__. 


This challenge provides a scaling roadmap for SKQD-based Quantum Chemistry experiments implemented on real quantum hardware from small-scale, classically simulable cases to large-scale, utility grade demonstrations. By the successful completion of all stages, you will be at the liminal boundary of State-of-the-Art, Utility-Scale Quantum Chemistry experimentation on Quantum Computers.

<div class="alert alert-block alert-success">

# Challenge instructions: SKQD

### Your tasks

Implement code to reproduce the results from the paper Quantum-Centric Algorithm for Sample-Based Krylov Diagonalization (https://arxiv.org/pdf/2501.09702).
Look for the `# PROMPT` marker inside the code boxes to find the places where code needs to be filled in. Where needed, there are specific `# BEGIN ANSWER` and `# END ANSWER` markers as well.

Specifically, this notebook is divided into two parts:

- __Part 1__: build and validate an SKQD workflow for the perturbed transverse-field Ising model (TFIM)
- __Part 2__: build a scalable SKQD workflow for the single-impurity Anderson model (SIAM)

## Grading system (100 points):

### Ex.1: Construct the perturbed TFIM Hamiltonian (10 points)
### Ex.2: Generate Krylov circuits for the TFIM (10 points)
### Ex.3: Execute the TFIM SKQD workflow and estimate the ground-state energy (30 points)
### Ex.4: Construct the SIAM Hamiltonian in the momentum basis (10 points)
### Ex.5: Generate Krylov circuits for the SIAM (20 points)
### Ex.6: Implement the SKQD SIAM workflow and compare against reference (20 points)
<div>

### **What directions are participants encouraged to explore?**

- Experiment with error suppression techniques (e.g., Pauli twirling, dynamical decoupling) through the `SamplerV2` primitive
- Find the most effective mapping of the Krylov circuits to your specific device via transpilation pass managers
- Investigate the impacts of noise, and how it affects bitstring quality
- Determine best experimental parameters, e.g., "samples per batch" 
- Explore effect of subspace dimensionality on the results
- Optimizing post-processing

## Python imports

We assume that we start from a Python environment that has been initialized by following the instructions in `Guide_and_install.ipynb`.

In [1]:
# Standard library imports
import numpy as np
import matplotlib.pyplot as plt
import scipy

# Qiskit core imports
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import BitArray

# Qiskit runtime imports
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2 as Estimator, SamplerV2 as Sampler

# Qiskit Aer imports
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import SamplerV2 as AerSampler

# Qiskit addons
from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.qubit import solve_qubit, sort_and_remove_duplicates

# Qiskit serverless
from qiskit_ibm_catalog import QiskitFunction, QiskitServerless

# Local helper module
import skqd_helpers


In [2]:
# Load your IBM Account

service = QiskitRuntimeService()

# PROMPT Define your prefered backend 

backend = service.least_busy(min_num_qubits=156)

print(f"Using backend {backend.name}")

Using backend ibm_kingston


Import the grader functions:

In [3]:
from qc_grader.challenges.r2p_2026_us.lab_skqd import (
    grade_lab_skqd_ex1,
    grade_lab_skqd_ex2,
    grade_lab_skqd_ex3,
    grade_lab_skqd_ex4,
    grade_lab_skqd_ex5,
    grade_lab_skqd_ex6,)
from qc_grader.challenges.r2p_2026_us import check_progress

<a id="p1"></a>
## **Part 1**<br>Solving the perturbed transverse-field Ising model using SKQD

Consider the perturbed transverse Ising model,

$$
\begin{equation}
H = -J\sum_{j=0}^{n-2} \sigma_j^z \sigma_{j+1}^z - h_x \sum_{j=0}^{n-1} \sigma_j^x - h_z \sigma_0^z
\tag{1},
\end{equation}
$$

where $\sigma_j^i$ correspond to the Pauli matrices defined as,

$$
\begin{equation}
\sigma^x =
\begin{bmatrix}
0 & 1 \\
1 & 0 \\
\end{bmatrix},~
\sigma^y =
\begin{bmatrix}
0 & -i \\
i & 0 \\
\end{bmatrix},~
\sigma^z =
\begin{bmatrix}
1 & 0 \\
0 & -1 \\
\end{bmatrix},
\tag{2}
\end{equation}
$$

acting on qubit at index $j$, $J$ is the coupling strength between neighboring spins, and $h_{\{x,z\}}$ are strengths of the applied field strengths in the ${\{x, z\}}$ direction.

<a id="ex1_1"></a>
<div class="alert alert-block alert-warning">

## **Ex.1**: Construct the perturbed TFIM Hamiltonian (10 points)

</div>

Using the provided function skeleton `perturbed_tfim_hamiltonian` below, define a method to construct the perturbed transverse-field Ising model (TFIM) Hamiltonian defined above as a [**`SparsePauliOp`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp).

In case you're unfamiliar with how to construct a [**`SparsePauliOp`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp) from a Hamiltonian, you can refer to the following:
  - [Quantum Simulation](https://quantum.cloud.ibm.com/learning/en/courses/utility-scale-quantum-computing/quantum-simulation#2-mapping-your-problem) (IBM Quantum Platform)
  - [Quantum Real Time Evolution using Trotterization](https://qiskit-community.github.io/qiskit-algorithms/tutorials/13_trotterQRTE.html) (qiskit-community)
  - [Coding a Hamiltonian in Qiskit](https://quantumcomputing.stackexchange.com/a/39670) (StackExchange)

In [7]:
# PROMPT: complete the function to construct the perturbed transverse-field Ising model
def perturbed_tfim_hamiltonian(
    num_qubits: int, J: float, hx: float, hz: float
) -> SparsePauliOp:
    """Builds the perturbed transverse-field Ising model Hamiltonian as a `SparsePauliOp`.

    Args:
        num_qubits: Number of qubits
        J: Exchange energy
        hx: Field strength in x-direction
        hz: Field strength in z-direction

    Returns:
        Hamiltonian as a SparsePauliOp following Qiskit's endian convention
    """

    paulis = []
    coeffs = []
    # BEGIN ANSWER
    # ZZ coupling between neighbors,  -J * sum_{j=0}^{n-2} Z_j Z_{j+1}
    for j in range(num_qubits - 1):
        label = ["I"] * num_qubits
        label[j]     = "Z"   # qubit j
        label[j + 1] = "Z"   # qubit j+1
        paulis.append("".join(label))
        coeffs.append(-J)
    # transverse field,  -hx * sum_{j=0}^{n-1} X_j
    for j in range(num_qubits):
        label = ["I"] * num_qubits
        label[j] = "X"
        paulis.append("".join(label))
        coeffs.append(-hx)
    # longitudinal perturbation on qubit 0 only
    label = ["I"] * num_qubits
    label[0] = "Z"
    paulis.append("".join(label))
    coeffs.append(-hz)


    # END ANSWER

    return SparsePauliOp(paulis, coeffs=np.array(coeffs, dtype=np.complex64))

#### NOTE: Testing and debugging
Run the cell below to check your implementation of **`perturbed_tfim_hamiltonian()`** defined in __Ex.1__. There should be no errors resulting from the assertions.

In [8]:
# Construct example Hamiltonian
test_hamiltonian = perturbed_tfim_hamiltonian(num_qubits=6, J=0.1, hx=0.2, hz=0.3)
paulis = test_hamiltonian.paulis
coeffs = test_hamiltonian.coeffs

# Expected Pauli terms
expected_paulis = ['ZZIIII', 'IZZIII', 'IIZZII', 'IIIZZI',
                    'IIIIZZ', 'XIIIII', 'IXIIII', 'IIXIII',
                    'IIIXII', 'IIIIXI', 'IIIIIX', 'ZIIIII']
# Expected Pauli coefficients
expected_coeffs = np.array([-0.1+0.j, -0.1+0.j, -0.1+0.j, -0.1+0.j,
                            -0.1+0.j, -0.2+0.j, -0.2+0.j, -0.2+0.j,
                            -0.2+0.j, -0.2+0.j, -0.2+0.j, -0.30000001+0.j])

# Checks
assert(paulis == expected_paulis)
assert(np.allclose(coeffs, expected_coeffs))


Throughout this challenge, you may want to add additional cells, such as the one above, to perform sanity checks of your own.

<div class="alert alert-block alert-success">

#### **Submit**
Run the cell below to submit your solution and compare against the reference.

In [9]:
# verify your answer
grade_lab_skqd_ex1(perturbed_tfim_hamiltonian)

Grading your answer. Please wait...

Congratulations! 🎉 Your answer is correct.
You scored 10 on this exercise.


<a id="ex1_2"></a>
<div class="alert alert-block alert-warning">

## **Ex.2**: Generate Krylov circuits for the TFIM (10 points)

</div>

Define the function, **`construct_krylov_circuits()`**, that constructs the Krylov (time evolution) circuits from the operator $U = \text{e}^{-iHt}$.
Implementing this operation exactly, however, generally requires an exponential number of gates.

As such, the evolution operators themselves should be of type [**`PauliEvolutionGate`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.PauliEvolutionGate). An important argument here is **`synthesis`** which implements an approximation of the unitary operator. There are a number of [**`qiskit.synthesis`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit/synthesis#module-qiskit.synthesis) methods available.

A common synthesis technique that results in low circuit depth is the Lie-Trotter product formula. This approximates the exponential of non-commuting operators with products of their exponentials up to error of second order:

$$
\begin{equation}
e^{A + B} \approx e^{A} e^{B}.
\end{equation}
$$

Feel free to experiment with different synthesis methods to reduce circuit depth.

In [12]:
from qiskit.circuit.library import PauliEvolutionGate
# PROMPT: fill this function
def construct_krylov_circuits(
        H_spo: SparsePauliOp,
        psi: QuantumCircuit,
        r: int,
        num_trotter_steps: int,
        dt: float
) -> list[QuantumCircuit]:
    """Build r Krylov circuits using time evolution generated by SparsePauliOp.

    Args:
        H_spo: Hamiltonian as SparsePauliOp
        psi: Initial state to time evolve
        r: number of Krylov basis states (i.e., Krylov dimension)
        num_trotter_steps: Number of Trotter steps per evolution gate
        dt: time step

    Returns:
        List of QuantumCircuits representing each Krylov state
    """
    # Time evolution operator: U = exp(-iH dt)
    circuits = []
    num_qubits = psi.num_qubits
    # BEGIN ANSWER
    gate = PauliEvolutionGate(H_spo, time=dt, synthesis=LieTrotter(reps=num_trotter_steps))
    circuits.append(psi.copy())

    for _ in range(1,r):
        psi.append(gate, range(num_qubits))
        circuits.append(psi.copy())
    
    # END ANSWER
    return circuits

<div class="alert alert-block alert-success">

#### **Submit**
Run the cell below to submit your solution and compare against the reference.
You can safely ignore any warnings of the type **`SparseEfficiencyWarning`**.

In [13]:
# verify your answer
grade_lab_skqd_ex2(construct_krylov_circuits, perturbed_tfim_hamiltonian)

Grading your answer. Please wait...

Congratulations! 🎉 Your answer is correct.
You scored 10 on this exercise.


<a id="ex1_3"></a>
<div class="alert alert-block alert-warning">

## **Ex.3**: Execute the TFIM SKQD workflow and estimate the ground-state energy (30 points)

</div>

Complete [**Part 1**](#part1) by working through each of the following cells. In the end, you will have found the approximate ground state energy which you will compare to the exact ground state energy.

This final exercise in Part 1 comprises of several sub-exercises but will be graded collectively.
Your score will depend on the precision achieved relative to the result obtained via exact diagonalization.

<a id="ex1_3_1"></a>
#### **Ex.3.1 Construct the Hamiltonian**<br>
Set the parameters of the Hamiltonian as follows:

```python
num_qubits = 12
J = 1.0
hx = 0.1
hz = 0.1
```

We chose a system small enough system such that it can be classically simulated and diagonalized within a reasonable timeframe on a modern laptop.

In [15]:
# PROMPT
# BEGIN ANSWER
# Hamiltonian parameters
num_qubits = 12
J = 1.0
hx = 0.1
hz = 0.1

# Construct Hamiltonian
H_spo = perturbed_tfim_hamiltonian(num_qubits=num_qubits, J=J, hx=hx, hz=hz)

# END ANSWER

<a id="ex1_3_2"></a>
#### **Ex.3.2 Construct the initial state and Krylov circuits**<br>
The Krylov space $\mathcal{K}^r$ of order $r$ is the space spanned by the vectors obtained by multiplying increasingly higher powers of a matrix $A$, up to $r-1$, with a vector $|v\rangle$:

$$
\begin{equation}
    \mathcal{K}^r = \{ A^0 |v\rangle, A^1 |v\rangle, ..., A^{r-1} |v\rangle \}.
\end{equation}
$$

In our case, $A \equiv U = e^{-iHdt}$, in which case the Krylov space is given as,

$$
\begin{equation}
    \mathcal{K}_U^r = \{ U^0 |v\rangle, U^1 |v\rangle, ..., U^{r-1} |v\rangle \}.
\end{equation}
$$

We will start with the initial state $|v\rangle \equiv \psi = |0^n\rangle$. When generating the Krylov circuits, note that the larger Krylov dimension $r$, the better approximation to the true ground state energy can be achieved as this leads to more of the full Hilbert space will be explored.
In our case, the full Hilbert space is quite small, $2^{12} = 4096$, so a Krylov dimension $r \sim 5$ should suffice.
Also keep in mind, especially when executing jobs on real hardware, the circuit depths required for Krylov circuits with large $r$.

An important parameter to set is the **time step**, or $dt$.
A theoretical result from [**Epperly _et al._**](https://arxiv.org/abs/2110.07492) showed that an optimal value for a time step is $\pi / ||H||$.
However, in this sampling-based context, the optimal choice for the time step is a topic of ongoing study; e.g., determining the time step heuristically, one may achieve better results by scaling $\pi / ||H||$ by some $O(1)$ factor.

Additionally, as there is no exact choice of Krylov dimension to choose, you may experiment with several here using [**`AerSimulator`**](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html) before starting [**Part 2**](#p2) where you will run on real quantum hardware.

Note you may receive a harmless **`ComplexWarning`** due to a cast from imaginary to real values. This can be safely disregarded.


In [21]:
# PROMPT
# BEGIN ANSWER
# Calculate an optimal dt for Trotterization using the
# `dt_from_spectral_norm()` helper function.

def dt_from_spectral_norm(Ham, factor=1.0):
    spectral_norm = np.linalg.norm(Ham.to_matrix(), ord=2)
    
    return factor * np.pi / spectral_norm

dt = dt_from_spectral_norm(H_spo)


# Specify a Krylov dimension (i.e., subspace size)
r = 5

# Construct the QuantumCircuit representing the intial state
num_qubits = H_spo.num_qubits
psi = QuantumCircuit(num_qubits)

# Construct the Krylov circuits
num_trotter_steps = 1
krylov_circuits = construct_krylov_circuits(H_spo, psi, r, num_trotter_steps, dt)
# END ANSWER

<a id="ex1_3_3"></a>
#### **Ex.3.3 Set backend and transpile circuits**<br>
Select the [**`AerSimulator`**](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html) backend and use [**`generate_preset_pass_manager()`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.generate_preset_pass_manager) to transpile the Krylov circuits to [**ISA circuits**](https://www.ibm.com/quantum/blog/isa-circuits) specific to backend.
While there are many arguments one can provide to generate a preset [**`PassManager`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.transpiler.PassManager), here we use a classical simulator so most can be ignored until [**Part 2**](#p2).



Note transpilation takes $O(1)$ seconds with an Apple M1 Max chip.

In [24]:
# PROMPT
# BEGIN ANSWER
from qiskit_aer import AerSimulator

for qc in krylov_circuits:
    qc.measure_all()
    
backend = AerSimulator()

# Create a pass manager for transpilation
pass_manager = generate_preset_pass_manager(optimization_level=1, backend=backend)

# Transpile all Krylov circuits to ISA circuits for the chosen backend
isa_circuits = pass_manager.run(krylov_circuits)

# END ANSWER

<a id="ex1_3_4"></a>
#### **Ex.3.4 Execute the job using `SamplerV2`**<br>
As the name implies, SKQD is a sampling-based algorithm as opposed to measuring physical observables within a quantum circuit.
The reason we want to work directly with samples is for the flexibility when using post-processing techniques that leverage, e.g., certain symmetries manifest in the system under study.
These symmetries can be used to filter out or 'correct' bad bitstrings.

As such, we will use a [**`SamplerV2`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/sampler-v2) object to execute the circuits and retrieve the results.
Classical sampling with [**`AerSimulator`**](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html) takes $O(10)$ seconds with an Apple M1 Max chip for modestly sized configurations of the solution.

In [26]:
# PROMPT
# BEGIN ANSWER
from qiskit_ibm_runtime import SamplerV2

num_shot = 100

# Instantiate a sampler to generate samples from the ISA circuits using the AerSimulator.
sampler = SamplerV2(mode=backend)

# Perform the classical simulation (run the job with the AerSimulator)
job = sampler.run(isa_circuits, shots=num_shot)


# Retrieve results from the job
result = job.result()


# END ANSWER

<a id="ex1_3_5"></a>
#### **Ex.3.5 Get bitstrings, probabilities**<br>
Using the [**`counts_to_arrays()`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-sqd/counts#counts_to_arrays) method from [**`qiskit-add-sqd`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-sqd), get the bitstrings.
We will need these for projection in the next step.

In [27]:
# PROMPT
# BEGIN ANSWER
merged_counts = {}
for pub_result in result:
    counts = pub_result.data.meas.get_counts()   # 'meas' from measure_all()
    for bitstring, freq in counts.items():
        merged_counts[bitstring] = merged_counts.get(bitstring, 0) + freq

# Get the bitstrings (and, optionally, probabilities)
bitstrings, probs = counts_to_arrays(merged_counts)

# END ANSWER

<a id="ex1_3_6"></a>
#### **Ex.3.6 Calculate ground state energy**<br>
Projection requires the bitstring matrix to be sorted and ascending in order by its unsigned integer representation.
For this, [**`qiskit-add-sqd`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-sqd)'s [**`sort_and_remove_duplicates()`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-sqd/qubit#sort_and_remove_duplicates) can be utilized. To calculate the eigenvalues and eigenvectors, [**`solve_qubit()`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-sqd/qubit#solve_qubit) from the same package will be useful.

In [29]:
# PROMPT
# BEGIN ANSWER

# Sort bitstring matrix and remove duplicates
bitstrings = sort_and_remove_duplicates(bitstrings)

# Calculate eigenvalues (and, optionally, eigenvectors)
energies, eigenstates = solve_qubit(bitstrings, H_spo, k=1, which="SA")

# END ANSWER

<a id="ex1_3_7"></a>
#### **Ex.3.7 Ground state energy from SKQD**<br>
Finally, determine ground state energy (i.e., minimum eigenvalue).

In [31]:
# PROMPT: Determine SKQD approximation to the ground state energy
# BEGIN ANSWER
est_min_eigenvalue = np.real(energies)

# END ANSWER

<div class="alert alert-block alert-success">

#### **Submit**
Execute the following cell to check your energy estimate against the reference, calculated using exact diagonalization.
The grader for this exercise takes in the Hamiltonian you constructed in __Ex.3.1__ and the bitstrings you obtained in __Ex.3.5__ from the `counts_to_array()` function.

In [32]:
# `bitstrings` is the variable which stores the first element returned by `counts_to_arrays()`
grade_lab_skqd_ex3(bitstrings, H_spo)

Grading your answer. Please wait...

Congratulations! 🎉 Your answer is correct.
You scored 20 on this exercise.


### **End of Part 1**
Congratulations on making it this far! Note you should have achieved an error $O(10^{-3})$ or better. If this is not the case, time permitting, go back and tweak some of the configuration (e.g., the number of Krylov basis states) to try getting a better SKQD estimate.

<a id="p2"></a>
## **Part 2: SKQD workflow in a Quantum-centric Supercomputing (QCSC) Environment**<br>

In [**Part 1**](#part1), we explored how sampled-based Krylov quantum diagonalization works and hopefully learned what parameters give the highest quality basis states for diagonalization. Helper methods are provided in the **`skqd_helpers`** module that we imported in [**Part 0**](#part0).

Now, we will implement a full-fledged SKQD workflow, as illustrated in the figure below.
For this, we will be using configuration recovery -- a technique that helps one *recover* expected bitstrings from those affected by device noise -- and diagonalizing in a QCSC environment with [**Qiskit Serverless**](https://quantum.cloud.ibm.com/docs/en/guides/serverless).

Qiskit Serverless is a framework for running distributed quantum and classical workloads without managing infrastructure. This means there is no server provisioning (no spinning up EC2s, clusters, Docker containers), no orchestration tools (Kubernetes, Docker Swarm), no monitoring/maintenance, etc. Each job that is run through Serverless runs in a clean container, executes your code, and then shuts down. There is no memory between jobs. You just write your code, define dependencies, and submit your job.

Qiskit Serverless gives a user access to always-on remote CPU cores and memory, allowing certain classical workloads to be distributed across the remote resources and to gain some advantage in parallel program processing while avoiding common headaches from device shutdown mid-execution.

In summary, Qiskit Serverless has these benefits:

1) Is it an always-on remote environment.
2) It has multiple remote CPU cores, allowing us to distribute computation efficiently across parallel resources.


The diagram below shows the architecture of QCSC for the SKQD workflow:

<img src="fig/qcsc-arch.png" alt="qcsc-arch"/>

### Single-impurity Anderson model (SIAM):

Our goal in Part 2 is to find the ground state energy of a chemistry Hamiltonian for the single-impurity Anderson model. It is of the form:

$$
\begin{equation}
\hat{H} = \sum_{p r, \sigma} h_{pr} \hat{a}_{p \sigma} \hat{a}_{r \sigma}
            + \sum_{p r q s, \sigma \tau} \frac{(pr| qs)}{2} \hat{a}_{p \sigma}^\dagger
            \hat{a}_{q \tau}^\dagger \hat{a}_{s \tau} \hat{a}_{r \sigma},
\end{equation}
$$

where $\hat{a}_{p \sigma}^\dagger$ ($\hat{a}_{p \sigma}$) is the creation (annihilation) operator associated to the $p$-th basis set element and spin $\sigma$, and $h_{pr}$ and $(pr|qs)$ are the one- and two-body electronic integrals (see Eq. 6 in [**this paper**](https://arxiv.org/abs/2405.05068) for more details) which can be obtained from standard chemistry software, such as [**PySCF**](https://github.com/pyscf/pyscf).

In this challenge, you will not be required to determine these integrals by yourself.

<a id="ex2_1"></a>
<div class="alert alert-block alert-warning">

## **Ex.4**: Construct the SIAM Hamiltonian in the momentum basis (10 points)

</div>


One of the main requirements for a successful application of the SKQD framework is that the Hamiltonian is sparse; in our case, when the bath is diagonal.
Since this is not the case in the position basis, we must rotate the orbitals in such a way as to perform a Fourier transform into the momentum basis - where the Hamiltonian is sparse.

As the construction of this Hamiltonian is quite involved and likely requires some background in quantum chemistry, we provide three helper methods in the **`skqd_helpers`** module:
- **`siam_hamiltonian()`**
- **`momentum_basis()`**
- **`rotated()`** 

You will need to use these three helper functions to construct the SIAM Hamiltonian one- and two-body terms in the momentum basis.

In [35]:
from skqd_helpers import siam_hamiltonian, momentum_basis,rotated

In [36]:
# PROMPT: complete this function
def siam_hamiltonian_momentum(
    num_orbs: int,
    hopping: float,
    onsite: float,
    hybridization: float,
    chemical_potential: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Hamiltonian for the single-impurity Anderson model (SIAM).

    Args:
        num_orbs: Number of spatial orbitals
        hopping: Hopping term
        onsite: On-site repulsion
        hybrid: Hybridization interaction
        mu: Chemical potential

    Returns:
        One- and two-body terms of Hamiltonian in momentum space.
    """
    # BEGIN ANSWER
    # Generate Hamiltonian in position basis
    h1e, h2e = siam_hamiltonian(num_orbs, hopping, onsite, hybridization, chemical_potential)
    
    # Rotate to momentum basis
    orbital_rotation = momentum_basis(num_orbs) 

    # Return rotated Hamiltonian
    rotated_hamiltonian = rotated(h1e, h2e, orbital_rotation.T.conj())

    # END ANSWER

    return rotated_hamiltonian

<div class="alert alert-block alert-success">

#### **Submit**
Execute the cell below to check your implementation of the SIAM Hamiltonian.

In [37]:
grade_lab_skqd_ex4(siam_hamiltonian_momentum)

Grading your answer. Please wait...

Congratulations! 🎉 Your answer is correct.
You scored 10 on this exercise.


<a id="ex2_2"></a>
<div class="alert alert-block alert-warning">

## **Ex.5**: Generate Krylov circuits for the SIAM (20 points)

</div>

As we did in __Ex.2.__ of Part 1, here we also need to construct the Krylov basis.
Since we are using explicitly the one- and two-body terms representing the SIAM Hamiltonian, this construction will be slightly different; specifically, when constructing a Trotter step for the circuits.
The provided **`trotter_step()`** method in the **`skqd_helpers`** module will take care of the complexities for you.
However, time permitting, you are encouraged to check out that method and understand its workings.

For the initial state preparation, we also provide **`sqkd_helpers.prepare_initial_state()`** to  assist in completing the method definition below that will generate the Krylov circuits. 

In [56]:
import scipy
from skqd_helpers import prepare_initial_state, trotter_step

In [73]:
def construct_krylov_siam(
    num_orbs: int,
    impurity_index: int,
    hamiltonian: tuple[np.ndarray, np.ndarray],
    dt: float,
    krylov_dim: int
) -> list[QuantumCircuit]:
    """Generate Krylov circuits for SIAM.

    Args:
        num_orbs: Number of spatial orbitals
        hamiltonian: one- and two-body Hamiltonian terms
        dt: Time step
        krylov_dim: Number of Krylov basis states

    Returns:
        SIAM Krylov circuits.
    """
    # BEGIN ANSWER 
    qubits = QuantumRegister(2 * num_orbs)

    # hamiltonian
    h1e_momentum, h2e_momentum = hamiltonian
    
    # quantum circuits
    qc = QuantumCircuit(qubits)

    # similar to the circuit construction as in part I
    # circuit structure in the inital state
    structure_set_initial = prepare_initial_state(qubits, num_orbs, num_orbs // 2)
    for structure in structure_set_initial:
        qc.append(structure)
    qc.measure_all()

    # initial "raw" circuit
    circuit_list = [qc.copy()]

    # add more and more steps
    
    one_step_norm = scipy.linalg.expm((-1.0j) * dt * h1e_momentum)
    trotter_ham = (one_step_norm, h2e_momentum)

    for _ in range(krylov_dim-1):
        qc.remove_final_measurements()
        structure_set_trotter_step = trotter_step(qubits, dt, trotter_ham, impurity_index, num_orbs)

        for structure in structure_set_trotter_step:
            qc.append(structure)

        qc.measure_all()
        circuit_list.append(qc.copy())

    
    # END ANSWER 
    return circuit_list

<div class="alert alert-block alert-success">

#### **Submit**
Check your implementation and receive your score by running the cell below.

In [74]:
help(trotter_step)

Help on function trotter_step in module skqd_helpers:

trotter_step(qubits: Sequence[qiskit.circuit.Qubit], time_step: float, hamiltonian: tuple[numpy.ndarray, numpy.ndarray], impurity_index: int, num_orbs: int)
    A Trotter step.
    
    Args:
        qubits: Ordered collection of qubits
        time_step: Time step
        hamiltonian: One- and two-body Hamiltonian terms,
        impurity_index: Index of impurity
        num_orbs: Number of spatial orbitals
    
    Returns:
        Trotter step generator.



In [76]:
# verify your answer
grade_lab_skqd_ex5(construct_krylov_siam, siam_hamiltonian_momentum)

Grading your answer. Please wait...

Congratulations! 🎉 Your answer is correct.
You scored 20 on this exercise.


<a id="ex2_3"></a>
<div class="alert alert-block alert-warning">

## **Ex.6**: Implement the SKQD SIAM workflow and compare against reference (20 points)

</div>

For this exercise, consider a half-filled system with a single impurity and the following parameters:

```python
num_orbs = 22
hybridization = 1.0
hopping = 1.0
onsite = 1.0
chemical_potential = -0.5 * onsite
```

Although we are using Qiskit Serverless, a QCSC-like environment, it is not yet a replacement for supercomputing platforms (e.g., ORNL's Frontier, NERSC's Perlmutter, etc.).
As such, we limit the number of orbitals, $n_\text{orbs}$, which corresponds to a number of qubits, $N = 2 n_\text{orbs}$, in this challenge.

As in __Ex.3.__, this exercise comprises multiple parts, and will be graded once all parts are completed.
Your score will depend on the precision achieved relative to the DMRG calculation.

<a id="ex2_3_1"></a>
#### **Ex.6.1 Map the problem to quantum circuits and operators**<br>
The SKQD workflow requires that there is polynomially overlap of the initial state with the true ground state.
However, while it is not always possible to know the true ground state in advance, there are useful ways that can help us get close.
In our case, we use [**Givens rotations**]() to prepare an initial state that is known to approximate the ground state of our model in the free fermion (i.e., non-interacting) model of our Hamiltonian.

<div style="text-align: center;">
    <img src="fig/initial-state.png" alt="initial-state" width="25%"/>
</div>

Despite our Hamiltonian involving interaction terms, this _ansatz_ serves well.

Using a combination of the methods defined in the previous exercises, as well as helper method **`prepare_initial_state()`**, construct circuits representing the Krylov basis states.
You can obtain a good time step using **`skqd_helpers.dt_from_spectral_norm()`** as before, but must choose a Krylov dimension appropriately.

In [116]:
# prameters
num_elec_a = 11
num_elec_b = 11
nelec = (num_elec_a, num_elec_b)
samples_per_batch = 100

In [117]:
# PROMPT
# BEGIN ANSWER
# Set Hamiltonian parameters
num_orbs = 22
hybridization = 1.0
hopping = 1.0
onsite = 1.0
chemical_potential = -0.5 * onsite

# Construct SIAM Hamiltonian
hamiltonian = siam_hamiltonian_momentum(num_orbs, hopping, onsite, hybridization, chemical_potential)
h1e, h2e = hamiltonian

# Construct Krylov circuits
# In the momentum basis the impurity sits at the center site (n_bath // 2)
impurity_index = (num_orbs - 1) // 2

# Good time step, and a Krylov dimension you choose
dt = float(8.0*np.pi / np.linalg.norm(h1e, ord=2))   # dt_from_spectral_norm does not work
#######
krylov_dim = 10   # change this if necessary

# Construct Krylov circuits (Ex.5)
krylov_circuits = construct_krylov_siam(num_orbs, impurity_index, hamiltonian, dt, krylov_dim)

# END ANSWER

In [118]:
print(samples_per_batch)

100


<a id="ex2_3_2"></a>
#### **Ex.6.2 Optimize for target hardware**<br>
In [**Part 1**](#p1), we focused on classical simulation using [**`AerSimulator`**](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html).
To execute a circuit on real quantum hardware, one must transpile the abstract circuit to the quantum instruction set architecture ([**ISA**](https://www.ibm.com/quantum/blog/isa-circuits)) for that device.
This ensure the circuit is represented in such a way that the backend hardware can understand, i.e., in the form of its native basis gates.

Select your backend and construct Krylov [**ISA circuits**](https://www.ibm.com/quantum/blog/isa-circuits).

In [119]:
# BEGIN ANSWER
# Imports
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Retrieve backend 
service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=100)
print(f"Using backend: {backend.name}")

# Construct optimized ISA circuits for the chosen backend
pass_manager = generate_preset_pass_manager(optimization_level=3, backend=backend)
isa_circuits = pass_manager.run(krylov_circuits)


# Construct optimized ISA circuits for backend
# END ANSWER

Using backend: ibm_kingston


<a id="ex2_3_3"></a>
#### **Ex.6.3 Execute on target hardware**<br>
Now that we have all the necessary ingredients, we are ready to execute the circuits on quantum hardware.

As before, use the Qiskit [**`SamplerV2`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/sampler-v2) primitive to generate samples from the [**ISA circuits**](https://www.ibm.com/quantum/blog/isa-circuits). Feel free to break this exercise up into several cells.

In [120]:
# BEGIN ANSWER
# Imports
from qiskit_ibm_runtime import SamplerV2 as Sampler

# Instantiate `SamplerV2` and set options
# Be sure to set SamplerV2.options.max_execution_time = 600 to stay within
# your allocation. Otherwise, your job may be terminated before completion.
sampler = Sampler(backend)
sampler.options.environment.job_tags = ["R2P_CAPSTONE"]

# Sample circuits
job = sampler.run(isa_circuits, shots=5000)
# END ANSWER

In [121]:
print(f"Job ID: {job.job_id()}")

Job ID: d93hscl958jc73bt20pg


In [ ]:
# Combine the counts from individual Trotter (Krylov) circuits
bit_array = BitArray.concatenate_shots(
    [result.data.meas for result in job.result()]
)

For later validation, save your samples -- stored above in `bit_array` -- to disk.

In [ ]:
np.savez('bit_array.npz', num_bits=num_orbs * 2, samples=bit_array.array)

<a id="ex2_3_4"></a>
#### **Ex.6.4 Post-process results**<br>
A key component to the SQKD workflow is post-processing, where bitstrings and their counts are used to perform **configuration recovery**: a technique that serves to mitigate (loosely, 'correct') errors introduced during circuit execution and state measurement.
The configuration recovery introduced in the paper, [**_Chemistry Beyond the Scale of Exact Diagonalization on a Quantum-centric Supercomputer_**](https://arxiv.org/abs/2405.05068) (beginning on page 16) gives the full details of a 'self-consistent' way to correct noisy bitstrings.
For our purposes, the figure below gives a high-level overview of how the algorithm works.
<div style="text-align: center;">
    <img src="fig/config-recovery.png" alt="configuration-recovery" width="35%"/>
</div>

In words, the algorithm proceeds as follows:
1. Input is a list of noisy sampled, $\tilde{\mathcal{X}} = \{ \mathbf{x} | \sim \tilde{P}_\Psi(\mathbf{x}) \}$, generated by a noisy quantum processor.
2. If particle number (a symmetry, or conservation, of our Hamiltonian) is not conserved, i.e., $N_\mathbf{x} \neq N$, $|N_\mathbf{x} - N|$ bits are flipped probabilistically to generate a new set $\tilde{\mathcal{X}}_R$ of _recovered configurations_.
3. Build $K$ batches of $d$ configurations, $\mathcal{S}^{(1)}, ..., \mathcal{S}^{(K)}$, from $\tilde{\mathcal{X}}_R$, distributed according to the frequencies of $\mathbf{x} \in \tilde{\mathcal{X}}_R$.
4. Obtain ground state energies are computed from $\hat{H}_{\mathcal{S}^{(K)}}$ which are then used to calculate updated occupancies.
5. Repeat until convergence.

Early versions of [**`qiskit-addon-sqd`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-sqd) required users implement much of this algorithm programmatically; thankfully, we now have a super straightforward way to run this entire part of the workflow -- configuration recovery, diagonalization, and calculate ground state energies -- with a single method call to [**`diagonalize_fermionic_hamiltonian()`**](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian).

To perform post-processing, we employ Qiskit Serverless for sufficient classical resources to handle the scale of our problem.
Using the template below, configure an instance of [**Qiskit Serverless**](https://quantum.cloud.ibm.com/docs/en/guides/serverless).
Provided for you is the engine to perform configuration recovery and diagonalization in the **`serverless`** directory -- you only need to correctly set up a client.
To help guide you, please refer to the section [**Deploy to IBM Quantum Platform**](https://quantum.cloud.ibm.com/docs/en/guides/serverless-first-program#deploy-to-ibm-quantum-platform) of the [**Qiskit Serverless**](https://quantum.cloud.ibm.com/docs/en/guides/serverless) documentation.

In [ ]:
# Instantiate a `QsikitServerless` object
client = QiskitServerless()

# PROMPT: upload the program intended to run to the Serverless cloud environment.
# (And re-upload it any time we change it's source code!)
# BEGIN ANSWER
from qiskit_serverless import QiskitFunction
diagonalize_function = QiskitFunction(
        title="diagonalization_engine",
        entrypoint="diagonalization_engine.py",
        working_dir="./serverless/",
    )
client.upload(diagonalize_function)
# END ANSWER

We are now ready to run the post-processing workflow.
This can be done either 'locally' on the system running **`jupyter-notebook`** or 'remotely' within the [**Qiskit Serverless**](https://quantum.cloud.ibm.com/docs/en/guides/serverless) environment set up above using the **`skqd_helpers.classically_diagonalize()`** method.

The following cell provides a basic configuration to execute the workflow.
Note setting **`local = True`** is a good debugging exercise, though should only be used for systems less than 30 qubits.
Additionally, you may want to experiment with different values of **`num_batches`** and **`samples_per_batch`** to achieve a better approximation to the ground state obtained via density matrix renormalization (DMRG).

In [ ]:
# `True` to run locally, `False` to run in Serverless environment
local = False

# SQD Options
energy_tol = 1e-4
occupancies_tol = 1e-3
max_iterations = 3  # Feel free to modify

# Eigenstate solver options
num_batches = 3  # Feel free to modify
samples_per_batch = 100  # Feel free to modify
max_cycle = 200
symmetrize_spin = True
carryover_threshold = 1e-5

When you're ready, make your call to **``skqd_helpers.classically_diagonalize()``** in the code cell below.

<a id="warning"></a>
<div class="alert alert-block alert-warning">

#### Running this cell on Qiskit Serverless may take up to 30 minutes, but it does not require any QPU time.

</div>

In [ ]:
# Call `skqd_helpers.classically_diagonalize()` and store the results
result = skqd_helpers.classically_diagonalize(
    # BEGIN ANSWER
    bit_array=bit_array,                       # bit array
    hcore=h1e,                           # 1-electron hamiltonian integrals
    eri=h2e,                             # 2-electron hamiltonian integrals
    num_orbitals=num_orbs,                    # Number of spatial orbitals
    nelec=num_orbs,                           # Number of electrons
    num_elec_a=num_orbs // 2,                      # Alpha orbitals
    num_elec_b=num_orbs // 2,                      # Beta orbitals
    job_id=job.job_id(),             # QPU bitstring Job ID
    client=client,                   # Diagonalization engine worker
    energy_tol=energy_tol,                      # SQD option
    occupancies_tol=occupancies_tol,                 # SQD option
    max_iterations=max_iterations,                 # SQD option
    num_batches=num_batches,                     # Eigenstate solver option
    samples_per_batch=samples_per_batch,               # Eigenstate solver option
    symmetrize_spin=symmetrize_spin,                 # Eigenstate solver option
    carryover_threshold=carryover_threshold,             # Eigenstate solver option
    max_cycle= max_cycle,                      # Eigenstate solver option
    mem=1,                           # Memory in GB (keep this fixed)
    local=local,                     # Remote vs Local Diagonalization
    # END ANSWER
)

<div class="alert alert-block alert-success">

#### **Submit**
Check how your SKQD workflow fares against DMRG by running the cell below.

In [ ]:
grade_lab_skqd_ex6(result, num_orbs, hopping, onsite, hybridization, chemical_potential)

### **End of Part 2**
We encourage you to continue experimenting and try improving your score.

# **End of SKQD Challenge**
Congratulations on completing the SKQD Challenge!
Thank you for your participation!